# Agent Inspection with AMBER

Deep-dive into how CliMaPan agents work with AMBER's columnar backend.
AMBER v0.3.1 keeps Python agent attributes and the DataFrame in sync automatically.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import ambr as am
from climapan_lab.base_params import economic_params
from climapan_lab.src.models import EconModel
from climapan_lab.src.consumers.Consumer import Consumer
from climapan_lab.src.firms.ConsumerGoodsFirm import ConsumerGoodsFirm
import polars as pl
import numpy as np

## 1. Create a model and inspect agent lists

In [ ]:
p = economic_params.copy()
p.update({'c_agents': 30, 'capitalists': 3, 'csf_agents': 2, 'cpf_agents': 1,
          'steps': 5, 'seed': 42, 'show_progress': False,
          'covid_settings': None, 'climateModuleFlag': False})

m = EconModel(p)
m.setup()  # setup() but don't run yet — inspect the agents

print('Agent lists created:')
print(f'  consumers:   {len(m.consumer_agents)}')
print(f'  CS firms:    {len(m.csfirm_agents)}')
print(f'  CP firms:    {len(m.cpfirm_agents)}')
print(f'  banks:       {len(m.bank_agents)}')
print(f'  government:  {len(m.government_agents)}')
if hasattr(m, 'greenEFirm'):
    print(f'  green energy:{len(m.greenEFirm)}')
    print(f'  brown energy:{len(m.brownEFirm)}')

## 2. Access individual agents

Each agent is a Python object with attributes synced to the DataFrame.

In [ ]:
# Pick the first consumer
c = m.consumer_agents[0]
print(f'Agent id={c.id}')
print(f'Type: {type(c).__name__}')
print(f'Age group: {c.getAgeGroup()}')
print(f'Consumer type: {c.getConsumerType()}')
print(f'Deposit: {c.deposit:.2f}')
print(f'Employed: {c.employed}')
print(f'Wage: {c.wage:.2f}')

## 3. Python attrs vs DataFrame — AMBER keeps them in sync

Setting `agent.wealth = 5` on a Python Agent auto-syncs to the columnar store.

In [ ]:
# Check the DataFrame before modification
df_before = m.agents_df.filter(pl.col('id') == c.id)
print('Deposit in DataFrame (before):', df_before['deposit'].item() if 'deposit' in df_before.columns else 'N/A')

# Modify the Python agent
c.deposit = 9999.0
print(f'Deposit on Python object: {c.deposit}')

# The DataFrame is synced after the next flush
m._flush_pending_writes()
df_after = m.agents_df.filter(pl.col('id') == c.id)
print('Deposit in DataFrame (after):', df_after['deposit'].item())

## 4. Filter agents with AMBER view API

In [ ]:
# Filter: only workers
workers = m.consumer_agents.select(
    m.consumer_agents.getConsumerType() == 'workers'
)
print(f'Workers in consumer list: {len(workers)}')

# Filter: only capitalists (owners)
owners = m.consumer_agents.select(
    m.consumer_agents.getConsumerType() == 'capitalists'
)
print(f'Capitalists in consumer list: {len(owners)}')

## 5. The full population DataFrame

All agents (consumers + firms + banks + government) share one DataFrame.

In [ ]:
pop = m.agents_df
print(f'Total population rows: {pop.height}')
print(f'Total columns: {len(pop.columns)}')
print(f'\nColumns with non-null values:')
for col in pop.columns[:20]:
    non_null = pop[col].drop_nulls().len()
    if non_null > 0:
        print(f'  {col}: {non_null} non-null')
print(f'  ... and {len(pop.columns) - 20} more columns')

## 6. Run a few steps and watch values change

In [ ]:
# Run 3 steps
for i in range(3):
    m.run_step()
    print(f'Step {m.t}: GDP={m.GDP:.1f}, employed={sum(1 for a in m.consumer_agents if a.employed)}')